## 🧹 Data Cleaning & Feature Engineering

Este notebook é responsável pela **limpeza, padronização e enriquecimento do dataset de reviews**, preparando os dados para as etapas posteriores de NLP e análise via LLM.

O foco principal não é apenas remover ruído, mas **garantir qualidade, consistência e viabilidade computacional** do pipeline.

---

### 📥 Carregamento e Inspeção Inicial

- Leitura do dataset original de reviews
- Inspeção das colunas, tipos de dados e valores ausentes
- Conversão de timestamps Unix para formato de data (`review_date`)

Colunas removidas nesta etapa:
- `Id` — identificador irrelevante para análise
- `ProfileName` — não utilizado no pipeline
- `Time` — substituído por `review_date`

---

### 🧠 Criação de Novas Features

Foram criadas features auxiliares para enriquecer a análise:

- **`helpfulness_score`**  
  Razão entre votos úteis e total de votos, representando a relevância percebida da review.

- **`review_length`**  
  Número de palavras da avaliação, utilizado para análise exploratória e controle de ruído.

- **`total_reviews`**  
  Quantidade total de avaliações por produto, usada para filtragem de produtos pouco representativos.

---

### 🧼 Limpeza de Texto

Principais tratamentos aplicados ao campo `Text`:

- Remoção de **tags HTML (`<span>`)**, comuns em reviews extraídas da web
- Normalização de espaços em branco
- Remoção de textos duplicados, preservando:
  - reviews com maior número de votos
  - maior `helpfulness_score`


Esse critério garante que versões mais relevantes de textos repetidos sejam mantidas.

---

### 🚫 Tratamento de Casos Problemáticos

- Reviews com `HelpfulnessDenominator = 0` foram removidas, evitando divisões inválidas
- Reviews extremamente curtas foram analisadas para identificação de ruído
- Produtos com **menos de 32 avaliações** foram descartados  

> 📌 **Motivação:**  
> Esse corte reduz drasticamente o volume de dados, mas preserva os produtos mais relevantes, tornando o preprocessamento de NLP e a vetorização computacionalmente viáveis.

---

### 📊 Resultado Final

Ao final do notebook, obtemos:

- Um dataset **limpo, consistente e sem duplicações**
- Features adicionais que enriquecem análises posteriores
- Um subconjunto de produtos com volume suficiente de dados
- Base preparada para:
  - preprocessamento de NLP
  - vetorização textual
  - agregações por produto
  - consumo por LLMs

O resultado final é salvo em:
**data/data_cleaned.csv**




### Primeiro contato com o dataset : 


In [1]:
import pandas as pd

data = pd.read_csv("../data/reviews.csv")

data.sample(5)

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
503554,503555,B006VFH0M4,A5W8LAYLOS26Q,"J. Trescott ""prime addict""",0,0,4,1349740800,"Delicious and full of flavor! Also, best deal ...",I became addicted to spanish ham while I was p...
421756,421757,B0016889I0,A33ZL7O110YLFH,MissNitha,1,1,4,1306454400,My Maypo...the best stuff on earth,I have loved this stuff since I was little. It...
16380,16381,B007TJGZ54,A3LCF0C8FPHOZ3,"Robert C. Reade ""Random buyer""",1,7,1,1297123200,Ridiculous delivery,"I ordered this coffee and another brand, it se..."
303427,303428,B001XQXZG6,A10KLMD5SHX22O,"Joseph A. Nahas ""Joeeno""",0,0,1,1322611200,Buyers beware!,The expiration on this box of tea which I purc...
428545,428546,B002R8J7YS,A16AXQ11SZA8SQ,"MamaBear007 ""MamaBear007""",6,6,5,1268179200,My dogs dance for these treats!,A dear friend sent these to my dogs as a gift....


### 📑 Descrição das Colunas
- **`Id`** - Identificador numérico do usuário.
- **`ProductId`** — Identificador único do produto.
- **`UserId`** — Identificador textual do usuário que escreveu a avaliação.
- **`ProfileName`** - Nome de perfil do usuário 
- **`HelpfulnessNumerator`** — Número de usuários que consideraram a review útil.
- **`HelpfulnessDenominator`** — Número total de usuários que votaram na utilidade da review.
- **`Score`** — Nota atribuída ao produto (1 a 5).
- **`Time`** - Timestamp horário da compra do registro
- **`Summary`** — Resumo curto da avaliação.
- **`Text`** — Texto completo da avaliação após limpeza.



Como funciona essa timestamp ?
O campo Time é um Unix Timestamp (ou Epoch time). É o número de segundos desde 01/01/1970 (UTC).

| Coluna                   | Significado                           |
| ------------------------ | ------------------------------------- |
| `HelpfulnessNumerator`   | Quantas pessoas acharam a review útil |
| `HelpfulnessDenominator` | Quantas pessoas votaram na review     |

**3 / 5 quer dizer que : 3 pessoas acharam util, 5 votaram.**



### Verificando dados faltantes :

In [2]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 568454 entries, 0 to 568453
Data columns (total 10 columns):
 #   Column                  Non-Null Count   Dtype
---  ------                  --------------   -----
 0   Id                      568454 non-null  int64
 1   ProductId               568454 non-null  str  
 2   UserId                  568454 non-null  str  
 3   ProfileName             568428 non-null  str  
 4   HelpfulnessNumerator    568454 non-null  int64
 5   HelpfulnessDenominator  568454 non-null  int64
 6   Score                   568454 non-null  int64
 7   Time                    568454 non-null  int64
 8   Summary                 568427 non-null  str  
 9   Text                    568454 non-null  str  
dtypes: int64(5), str(5)
memory usage: 43.4 MB


Podemos percebeber que temos dados faltantes em ProfileName e em summary.

### Transformando a timestamp em data :

In [3]:
data["review_date"] = pd.to_datetime(data["Time"], unit="s")

### Deletando colunas irrelevantes : 

In [4]:
data.drop(columns=["Id","ProfileName","Time"], inplace=True )

### Criando a feature helpfulness_score :

In [5]:
data["helpfulness_score"] = (
    data["HelpfulnessNumerator"] /
    data["HelpfulnessDenominator"]
)

In [6]:
data.sample(5, random_state=100)

,ProductId,UserId,HelpfulnessNumerator,HelpfulnessDenominator,Score,Summary,Text,review_date,helpfulness_score
238235,B001E5E1O6,A2QVIP2O0J08BU,0,0,5,wow!,great product. great price for the product. ...,2009-04-05,NaN
370991,B000UXZ1PQ,A1CRI3DKT18JBX,2,2,5,Fresh with consistent results,"Excellent value, with fresh grains, yielding c...",2011-05-11,1.000000
147634,B001D0IZBM,A2L24PZ59NJ3Q9,0,0,5,Fast shipping and excellent product,We enjoy the Keurig k-cup so much and there is...,2011-05-01,NaN
393369,B000W7PUOW,A2EC7QA31CFK4V,2,2,5,Planters Honey Peanuts,Iam very satisfied with the Planters honey pea...,2008-03-20,1.000000
399289,B0039KER1G,A25RGKC4ZEZ3LV,5,7,1,It is not mangosteen,It is not mangosteen.<br />I eat mangosteen fr...,2010-12-31,0.714286


Com a criação da feature **helpfulness_score**, é possível avaliar a relevância de cada review.  
Uma avaliação tende a ser mais informativa quando apresenta um **alto helpfulness_score**, indicando que foi considerada útil por uma grande proporção de usuários que interagiram com ela.

Reviews com valores ausentes (NaN) nessa feature possuem HelpfulnessNumerator e/ou HelpfulnessDenominator iguais a zero, o que significa que **não receberam votos de utilidade** ou **não foram validadas por outros usuários**.  
Por esse motivo, essas avaliações foram removidas da análise.


### Verificando os valores nulos de helpfulness_score:

In [7]:
data.helpfulness_score.info()

<class 'pandas.Series'>
RangeIndex: 568454 entries, 0 to 568453
Series name: helpfulness_score
Non-Null Count   Dtype  
--------------   -----  
298402 non-null  float64
dtypes: float64(1)
memory usage: 4.3 MB


Podemos perceber agora que tem apenas 298402 valores não nulos, isso vem do fato de que HelpfulnessNumerator e HelpfulnessDenominator possuem zeros (como explicado acima), **podemos descartar estes registros pois eles possuem pouca confiabilidade.**

In [8]:
data[data["HelpfulnessDenominator"] ==0].shape

(270052, 9)

In [9]:
data = data[data["HelpfulnessDenominator"] !=0]

Apagamos apenas o HelpfulnessDenominator pois se ele é 0, HelpfulnessNumerator também é

In [10]:
data.shape

(298402, 9)

Este é nosso total de registros agora.

### Verificando valores únicos :

In [11]:
data.nunique()

ProductId                  49316
UserId                    149991
HelpfulnessNumerator         231
HelpfulnessDenominator       233
Score                          5
Summary                   168010
Text                      209488
review_date                 3130
helpfulness_score            951
dtype: int64

In [12]:
data.Score.min()

np.int64(1)

Temos 298402 registros, 49316 produtos distintos, 149991 usuários, 231 valores distintos de HelpfulnessNumerator e 233 no denominator, score vai de 1 a 5, summary 168010 (repetidos?) Text 209488 (repetidos?)

### APagando valores nulos :

In [13]:
data.dropna()
data.info()

<class 'pandas.DataFrame'>
Index: 298402 entries, 0 to 568452
Data columns (total 9 columns):
 #   Column                  Non-Null Count   Dtype        
---  ------                  --------------   -----        
 0   ProductId               298402 non-null  str          
 1   UserId                  298402 non-null  str          
 2   HelpfulnessNumerator    298402 non-null  int64        
 3   HelpfulnessDenominator  298402 non-null  int64        
 4   Score                   298402 non-null  int64        
 5   Summary                 298376 non-null  str          
 6   Text                    298402 non-null  str          
 7   review_date             298402 non-null  datetime64[s]
 8   helpfulness_score       298402 non-null  float64      
dtypes: datetime64[s](1), float64(1), int64(3), str(4)
memory usage: 22.8 MB


### Criando a coluna review_length :

In [14]:
data["review_length"] = data["Text"].str.split().str.len()

data["review_length"].describe()


count    298402.000000
mean         89.660401
std          91.337398
min           3.000000
25%          36.000000
50%          63.000000
75%         109.000000
max        3432.000000
Name: review_length, dtype: float64

In [15]:

short_reviews = data[data["review_length"] < 10]

short_reviews[["Text", "review_length"]].head(20)


,Text,review_length
15560,Same price as Dr. Foster & Smith.,7
44455,"Yummy, if you like noodles I recomend these.",8
49025,"<span class=""tiny""> Length:: 1:02 Mins<br /><b...",7
65997,"<span class=""tiny""> Length:: 0:42 Mins<br /><b...",7
70602,Tastes good. Fruit juice sweetened. Excellent ...,7
80655,Great gift. Fast delivery. Quality product.,6
94052,Same price as Dr. Foster & Smith.,7
106960,Tastes great and the price was good.,7
107456,"<span class=""tiny""> Length:: 4:34 Mins<br /><b...",7
107554,Tastes great and the price was good.,7


Com esta nova coluna fica explícito que para reviews pequenas possuímos muitos spams, serão retirados.

### Retirando span class :

In [16]:
mask_span_class = data["Text"].str.contains(
    r"<span\s+class",
    regex=True,
    case=False,
    na=False
)

data[mask_span_class][["Text"]].head(10)


,Text
158,"<span class=""tiny""> Length:: 0:26 Mins<br /><b..."
1460,"<span class=""tiny""> Length:: 1:38 Mins<br /><b..."
5487,"<span class=""tiny""> Length:: 0:32 Mins<br /><b..."
11803,"<span class=""tiny""> Length:: 0:51 Mins<br /><b..."
14064,"<span class=""tiny""> Length:: 1:39 Mins<br /><b..."
14157,"<span class=""tiny""> Length:: 1:15 Mins<br /><b..."
14197,"<span class=""tiny""> Length:: 1:10 Mins<br /><b..."
15828,"<span class=""tiny""> Length:: 1:30 Mins<br /><b..."
16053,"<span class=""tiny""> Length:: 2:09 Mins<br /><b..."
19164,"<span class=""tiny""> Length:: 0:39 Mins<br /><b..."


In [17]:
mask_span_class.mean() * 100

np.float64(0.10254622958291164)

Span representa cerca de 10% das reviews.

#### Removendo :

In [18]:
import re

def remove_span_tags(text):
    return re.sub(r"</?span[^>]*>", " ", text)

data["Text"] = data["Text"].apply(remove_span_tags)


### Verificando avaliações duplicadas : 

In [19]:
dup_mask = data.duplicated(subset=["Text"], keep=False)

dup_mask.sum()


np.int64(117723)

In [20]:
data = (
    data
    .sort_values(
        ["HelpfulnessDenominator", "helpfulness_score"],
        ascending=False
    )
    .drop_duplicates(subset=["Text"], keep="first")
)


Antes de remover duplicatas, os dados foram ordenados pelo número total de votos
(HelpfulnessDenominator) e pelo helpfulness_score, garantindo que reviews mais
confiáveis e com maior engajamento apareçam primeiro.

Em seguida, duplicatas de texto foram removidas mantendo apenas a primeira
ocorrência, que corresponde à review com maior evidência social. Dessa forma,
evita-se viés causado por textos repetidos, reduz-se ruído no dataset e preserva-se
a versão mais representativa de cada review para as etapas de NLP.


In [21]:
data.duplicated(subset=["Text"]).sum()


np.int64(0)

In [22]:
data.shape

(209488, 10)

Temos agora um total de 209488 reviews.

In [23]:
data["Text"] = (
    data["Text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

data.head()

,ProductId,UserId,HelpfulnessNumerator,HelpfulnessDenominator,Score,Summary,Text,review_date,helpfulness_score,review_length
207712,B00012182G,A1JUGIQDY6UYSM,844,923,3,Whole Rabbit - NOT!,"I ordered one of these Fresh ""Whole"" Rabbits, ...",2009-09-08,0.914410,111
190733,B000FI4O90,A1GQGYROVZVW49,866,878,5,Works as Advertised - Classy Product,see update at end of review<br /><br />*******...,2006-11-28,0.986333,1223
566779,B001PQTYN2,A1QB2Y8GSME58Y,808,815,5,sauce not for mortals,I purchased a burrito from a small shop a few ...,2009-12-14,0.991411,346
235722,B001F10XUU,A39V22BIBUMMB3,580,593,1,Lost in Translation: Truth,"This product is called ""Hunmatsu-RyokuCha,"" in...",2011-07-02,0.978078,770
222937,B000UUWECC,AU3GYRAKBUAEU,491,569,3,"not bad stuff, but I have serious questions",Coconut water is the liquid inside an unopened...,2008-06-01,0.862917,708


Esse trecho realiza a normalização final do texto, removendo espaços em excesso, quebras de linha e espaços no início ou no fim das avaliações, garantindo maior consistência e qualidade dos dados antes das etapas de NLP.

### Criação da coluna total_reviews : 

In [24]:
data["total_reviews"] = (
    data
    .groupby("ProductId")["Score"]
    .transform("count")
)

data["total_reviews"].describe()


count    209488.000000
mean         31.745503
std          54.980779
min           1.000000
25%           4.000000
50%          11.000000
75%          32.000000
max         451.000000
Name: total_reviews, dtype: float64

Esta coluna representa a contagem total  de reviews para cada produto, Aqui, o Score não é usado pelo valor, mas sim como uma coluna base para contagem. ou seja, sempre que tiver score, é uma review.

In [25]:
data = data[data["total_reviews"] >= 32]


Para questões de otimização e tamanho dos dados, filtrei que apenas registros com 32 ou mais reviews sejam válidos para a análise.

In [26]:
data.shape

(52405, 11)

In [27]:
data.nunique()

ProductId                   795
UserId                    42366
HelpfulnessNumerator        208
HelpfulnessDenominator      210
Score                         5
Summary                   45012
Text                      52381
review_date                2616
helpfulness_score           717
review_length               774
total_reviews               139
dtype: int64

Ainda ficamos com 52405 registros e  42366 produtos distintos

### Salvando os dados após a limpeza :

In [28]:
data.to_csv("../data/data_cleaned.csv")